# Master Dataset Creation

In [1]:
import pandas as pd
import re

In [15]:
human_df = pd.read_csv("../data/human_dataset.csv")
#ai_df = pd.read_csv("../data/ai_generated_zeroshot.csv")
#ai_df = pd.read_csv("../data/ai_generated_fewshot.csv")
ai_df = pd.read_csv("../data/ai_generated_rewrites.csv")


print(human_df.shape)
print(ai_df.shape)

(600, 5)
(600, 8)


In [16]:
# colums check
print(human_df.columns)
print(ai_df.columns)


Index(['source_id', 'source', 'text', 'label', 'generation_type'], dtype='str')
Index(['source_id', 'source', 'text', 'label', 'generation_type', 'topic',
       'latency_sec', 'error'],
      dtype='str')


In [17]:
# check missing

print(human_df.isna().sum())
print(ai_df.isna().sum())

source_id          0
source             0
text               0
label              0
generation_type    0
dtype: int64
source_id            0
source               0
text                 0
label                0
generation_type      0
topic                0
latency_sec          0
error              600
dtype: int64


In [18]:
# remove non-ASCII characters from AI text

def clean_text(text):
    if pd.isna(text):
        return text
    return re.sub(r'[^\x00-\x7F]+', '', text)

ai_df["text"] = ai_df["text"].apply(clean_text)

In [19]:
# remove broken rows
human_df = human_df[human_df["text"].str.strip() != ""]
ai_df = ai_df[ai_df["text"].str.strip() != ""]

In [20]:
# check labels
print(human_df["label"].value_counts())
print(human_df["generation_type"].value_counts())
print(human_df["source"].value_counts())

print(ai_df["label"].value_counts())
print(ai_df["generation_type"].value_counts())
print(ai_df["source"].value_counts())

label
human    600
Name: count, dtype: int64
generation_type
human    600
Name: count, dtype: int64
source
articles.csv        200
yelp.csv            200
essays_texts.csv    200
Name: count, dtype: int64
label
ai    600
Name: count, dtype: int64
generation_type
ai_rewrite    600
Name: count, dtype: int64
source
news      200
essay     200
review    200
Name: count, dtype: int64


In [21]:
# drop AI only columns
ai_df = ai_df[["source_id", "source", "text", "label", "generation_type"]]

In [22]:
# merge data
df = pd.concat([human_df, ai_df], ignore_index=True)

In [23]:
# shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df["id"] = df.index # add id column

In [24]:
# reorder columns
df = df[["id", "text", "label","generation_type", "source", "source_id"]]

In [25]:
print("\nFinal Shape")
print(df.shape)

print("\n-Label Counts")
print(df["label"].value_counts())

print("\n-Source Counts")
print(df["source"].value_counts())


Final Shape
(1200, 6)

-Label Counts
label
ai       600
human    600
Name: count, dtype: int64

-Source Counts
source
news                200
essays_texts.csv    200
articles.csv        200
essay               200
yelp.csv            200
review              200
Name: count, dtype: int64


In [26]:
df.head(5)

,id,text,label,generation_type,source,source_id
0,0,Title: Waqar Younis Resigns as Pakistan Coach...,ai,ai_rewrite,news,news_538
1,1,"Kolkata, India - West Indies' mercenary squad...",ai,ai_rewrite,news,news_416
2,2,"on. A female lies on her back, calling like cr...",human,human,essays_texts.csv,essays_1388_chunk3
3,3,LONDON: Joe Root was named Test Player of the ...,human,human,articles.csv,articles_1754
4,4,strong>ISLAMABAD: Guinness Book of World Recor...,human,human,articles.csv,articles_1994


In [27]:
# save
df.to_csv("../data/master_rewrites_dataset.csv", index=False)